# parallel_executor\nGenerated from 00_common/parallel_executor.py for Databricks notebook import.\n

In [ ]:
"""Parallel execution engine for orchestrating multiple sources concurrently."""\n\nfrom __future__ import annotations\n\nimport logging\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom typing import Callable, Optional, Any\n\nlogger = logging.getLogger(__name__)\n\n\nclass ParallelExecutor:\n    """\n    Manages parallel execution of independent tasks with dependency tracking.\n    \n    Features:\n    - Configurable worker pool size\n    - Graceful error handling\n    - Progress tracking\n    - Timeout support\n    """\n    \n    def __init__(self, max_workers: int = 4):\n        """\n        Initialize ParallelExecutor.\n        \n        Args:\n            max_workers: Maximum number of concurrent workers\n        """\n        self.max_workers = max(1, min(max_workers, 16))  # Clamp between 1-16\n        self.results: dict[str, Any] = {}\n        self.errors: dict[str, Exception] = {}\n    \n    def execute_tasks(\n        self,\n        tasks: dict[str, tuple[Callable, tuple, dict]],\n        timeout_seconds: Optional[int] = None,\n    ) -> dict[str, Any]:\n        """\n        Execute multiple tasks in parallel.\n        \n        Args:\n            tasks: Dict of task_id -> (callable, args_tuple, kwargs_dict)\n            timeout_seconds: Timeout per task\n            \n        Returns:\n            Dict of task_id -> result\n        """\n        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:\n            # Submit all tasks\n            future_to_task = {\n                executor.submit(func, *args, **kwargs): task_id\n                for task_id, (func, args, kwargs) in tasks.items()\n            }\n            \n            # Process completed tasks as they finish\n            completed = 0\n            for future in as_completed(future_to_task, timeout=timeout_seconds):\n                task_id = future_to_task[future]\n                try:\n                    result = future.result(timeout=timeout_seconds)\n                    self.results[task_id] = result\n                    completed += 1\n                    logger.info(f"Task {task_id} completed ({completed}/{len(tasks)})")\n                except Exception as e:\n                    self.errors[task_id] = e\n                    logger.error(f"Task {task_id} failed: {e}")\n        \n        return self.results\n    \n    def get_results(self) -> tuple[dict[str, Any], dict[str, Exception]]:\n        """Get all results and errors."""\n        return self.results, self.errors\n    \n    def has_errors(self) -> bool:\n        """Check if any errors occurred."""\n        return len(self.errors) > 0\n